# Example 4: Creating a Coloured Globe (Complete Pipeline)

This notebook demonstrates the complete `globe3d` model generation pipeline:
1. Generate high-resolution outer sphere and inner sphere.
2. Apply ETOPO topography displacement to the outer shell.
3. Apply sharp coastline step boundary from a shapefile.
4. Split and hollow both hemispheres while adding magnet joint features.
5. Assign vertex colors to the outer shell using a seismic tomography dataset.
6. Assign a solid, neutral gray color to the inner hollow shell vertices using a color modifier.
7. Export both hemispheres to OBJ files with vertex colors.

## Step 1: Import libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import trimesh
from globe3d import (
    GlobeModel,
    GeographicGrid,
    GridDisplacer,
    LineDisplacer,
    GridColourer,
    ConstantColourer,
    calculate_displacement_scale
)

## Step 2: Generate outer and inner spheres

In [ ]:
model_radius_mm = 40.0
# Initialize GlobeModel
model = GlobeModel.from_fibonacci(n_points=8000, radius=model_radius_mm)
# Generate inner mesh (diameter reduction: 2 * (40.0 - 40.0 * 0.75) = 20.0 mm wall thickness)
model.create_inner_mesh(thickness=20.0)

## Step 3: Apply topography and coastline step

In [ ]:
# Load topo
topo_grid = GeographicGrid.from_netcdf("../inputs/ETOPO_2022_v1_60s_N90W180_surface.nc", 'lat', 'lon', 'z')
topo_grid_ds = GeographicGrid(
    lats=topo_grid.lats[::10],
    lons=topo_grid.lons[::10],
    grid=topo_grid.grid[::10, ::10]
)

# Displace topography
scale = calculate_displacement_scale(model_radius_mm, earth_radius_km=6371.0, vertical_exagg=40.0)
model.displace(GridDisplacer(topo_grid_ds), scale=scale)

# Coastline step
model.displace(LineDisplacer(
    shapefile_path="../inputs/coastlines/ne_110m_coastline.shp",
    displacement=0.8,
    width_degrees=0.5
))

## Step 4: Split, hollow, and insert magnets

In [ ]:
# Register MagnetSettings on model
from globe3d.magnets import MagnetSettings
model.magnet_settings = MagnetSettings(
    diameter=5.0,
    height=2.0,
    horizontal_tolerance=0.15,
    vertical_tolerance=0.10,
    vertical_offset=0.20,
    min_thickness=1.5,
    n_magnets=3,
    add_bosses=True
)

## Step 5: Apply independent surface coloring

We color the outer shell from a seismic tomography grid, and then we use the `modify_vertex_colors` modifier function with the `'inward_facing'` selection option to paint the inner cavity vertices a solid grey.

In [ ]:
# Load tomography grid
tomo_grid = GeographicGrid.from_netcdf("../inputs/s40_depth_slice_2850.grd", 'y', 'x', 'z')

# 1. Color outer (outward-facing) surfaces using tomography grid
tomo_colourer = GridColourer(tomo_grid, colormap='RdBu_r', vmin=-2.0, vmax=2.0)
model.colour(tomo_colourer, selection='outward_facing')

# 2. Color inner (inward-facing) cavity surfaces to gray
gray_colourer = ConstantColourer([0.6, 0.6, 0.6])
model.colour(gray_colourer, selection='inward_facing')

# 3. Generate hemispheres (colored from the model's registered recipe)
top_half, bottom_half = model.generate_hemispheres(
    hollow=True,
    thickness=20.0,
    engine='manifold'
)

## Step 6: Preview colors in 3D

In [ ]:
fig = plt.figure(figsize=(12, 6))

top_colors = top_half.visual.vertex_colors[:, :3].astype(float) / 255.0
bottom_colors = bottom_half.visual.vertex_colors[:, :3].astype(float) / 255.0

# Top half colored
ax1 = fig.add_subplot(121, projection='3d')
pts_top = top_half.vertices
sc1 = ax1.scatter(pts_top[:, 0], pts_top[:, 1], pts_top[:, 2], c=top_colors, s=2)
ax1.set_title("Colored Top Hemisphere")

# Bottom half colored
ax2 = fig.add_subplot(122, projection='3d')
pts_bot = bottom_half.vertices
sc2 = ax2.scatter(pts_bot[:, 0], pts_bot[:, 1], pts_bot[:, 2], c=bottom_colors, s=2)
ax2.set_title("Colored Bottom Hemisphere")

plt.show()

## Step 7: Export to OBJ with vertex colors

In [ ]:
output_dir = "../outputs"
os.makedirs(output_dir, exist_ok=True)

from globe3d.io import write_obj_with_vertex_colors

write_obj_with_vertex_colors(
    filename=os.path.join(output_dir, "example_4_top.obj"),
    vertices=top_half.vertices,
    faces=top_half.faces,
    colors=top_colors
)

write_obj_with_vertex_colors(
    filename=os.path.join(output_dir, "example_4_bottom.obj"),
    vertices=bottom_half.vertices,
    faces=bottom_half.faces,
    colors=bottom_colors
)
print("Colored OBJ hemispheres exported successfully!")